In [ ]:
CATALOGO = "desafio_vendas"
O = f"{CATALOGO}.ops"
G = f"{CATALOGO}.gold"

spark.sql(f"""
CREATE OR REPLACE VIEW {O}.vw_ultima_execucao AS
SELECT ordem, etapa, status, inicio, fim, ROUND(duracao_s, 1) AS duracao_s,
       qtd_lida, qtd_gravada, qtd_rejeitada, qtd_duplicada, id_lote_camada, detalhe, id_execucao
FROM {O}.log_execucao
WHERE id_execucao = (SELECT id_execucao FROM {O}.log_execucao ORDER BY inicio DESC LIMIT 1)
ORDER BY ordem
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {O}.vw_historico_execucoes AS
SELECT id_execucao, MIN(inicio) AS inicio, MAX(fim) AS fim,
       ROUND(UNIX_TIMESTAMP(MAX(fim)) - UNIX_TIMESTAMP(MIN(inicio)), 1) AS duracao_total_s,
       CASE WHEN SUM(CASE WHEN status <> 'SUCESSO' THEN 1 ELSE 0 END) = 0
            THEN 'SUCESSO' ELSE 'COM_PROBLEMAS' END AS status_execucao
FROM {O}.log_execucao GROUP BY id_execucao
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {O}.vw_funil_registros AS
WITH ult AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY etapa ORDER BY fim DESC) AS rn
  FROM {O}.log_pipeline)
SELECT etapa, tabela_destino, qtd_lida, qtd_gravada,
       COALESCE(qtd_rejeitada, 0) AS qtd_rejeitada, COALESCE(qtd_duplicada, 0) AS qtd_duplicada, status, fim
FROM ult WHERE rn = 1
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {O}.vw_qualidade_ultima AS
WITH ult AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY etapa, regra_id ORDER BY data_execucao DESC) AS rn
  FROM {O}.qualidade_resultados)
SELECT etapa, regra_id, descricao, severidade, qtd_afetada, qtd_total, pct_afetado, data_execucao,
       CASE WHEN qtd_afetada = 0 THEN 'OK'
            WHEN severidade = 'ERRO' AND etapa = 'silver' THEN 'QUARENTENA'
            WHEN severidade = 'ERRO' THEN 'CRITICO'
            ELSE 'ALERTA' END AS situacao
FROM ult WHERE rn = 1
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {O}.vw_score_qualidade AS
SELECT COUNT(*) AS registros_silver,
  ROUND(100.0 * SUM(CASE WHEN SIZE(alertas) = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_sem_alertas,
  ROUND(100.0 * SUM(CASE WHEN SIZE(FILTER(alertas, x -> x <> 'R08')) = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_sem_alertas_exceto_r08,
  ROUND(100.0 * SUM(CASE WHEN ARRAY_CONTAINS(alertas, 'R07') THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_retirada_inconsistente
FROM {CATALOGO}.silver.vendas
""")

for v in ["vw_ultima_execucao", "vw_historico_execucoes", "vw_funil_registros", "vw_qualidade_ultima", "vw_score_qualidade"]:
    print(f"\n=== {v} ===")
    display(spark.table(f"{O}.{v}"))

# Evolução das transformações: histórico de versões da Delta
display(spark.sql(f"DESCRIBE HISTORY {G}.fato_vendas"))